# 01 Data Gathering - 8 Dataset Terpisah

Notebook ini hanya melakukan **gathering** untuk 8 file raw yang ada di `data/raw/`. Setiap file diperlakukan sebagai dataset sendiri, disimpan sendiri, dan tidak dibuat dataset transaksi gabungan.

Output utama notebook ini:
- `data/interim/raw_separate/<dataset_id>.csv`
- `data/interim/raw_separate/raw_dataset_manifest.json`
- ringkasan visual untuk memastikan semua dataset sudah terbaca.

## Setup Environment
Menyiapkan library, path project, folder output, dan konfigurasi tampilan dataframe.

In [1]:
import json
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from IPython.display import display, HTML

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

try:
    from fingo_ds1.config import RAW_DATA_PATH, INTERIM_DATA_PATH, REPORTS_PATH
except ImportError:
    RAW_DATA_PATH = project_root / 'data' / 'raw'
    INTERIM_DATA_PATH = project_root / 'data' / 'interim'
    REPORTS_PATH = project_root / 'reports'

raw_data_path = Path(RAW_DATA_PATH)
interim_data_path = Path(INTERIM_DATA_PATH)
reports_path = Path(REPORTS_PATH)
raw_separate_path = interim_data_path / 'raw_separate'
figures_path = reports_path / 'figures'

raw_separate_path.mkdir(parents=True, exist_ok=True)
figures_path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)

print(f'Project root      : {project_root}')
print(f'Raw data path     : {raw_data_path}')
print(f'Raw separate path : {raw_separate_path}')

Project root      : /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Raw data path     : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw
Raw separate path : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim/raw_separate


## Dataset Catalog
Delapan dataset didefinisikan eksplisit berdasarkan isi folder `data/raw/`. Dataset e-commerce bulanan tetap diperlakukan sebagai 6 dataset terpisah.

In [2]:
DATASET_CATALOG = [
    {
        'dataset_id': 'ecommerce_2024_01_january',
        'dataset_name': 'E-Commerce Sales - January 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-01',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '01_JanuarySales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_06_june',
        'dataset_name': 'E-Commerce Sales - June 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-06',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '02_JuneSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_12_december',
        'dataset_name': 'E-Commerce Sales - December 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-12',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '03_DecemberSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_02_february',
        'dataset_name': 'E-Commerce Sales - February 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-02',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '04_FebruarySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_07_july',
        'dataset_name': 'E-Commerce Sales - July 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-07',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '05_JulySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_11_november',
        'dataset_name': 'E-Commerce Sales - November 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-11',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '06_NovemberSales2025_clean.xlsx',
    },
    {
        'dataset_id': 'daily_household_transactions',
        'dataset_name': 'Daily Household Transactions',
        'domain': 'household_finance',
        'dataset_period': None,
        'source_path': raw_data_path / 'daily_household_transaction' / 'Daily Household Transactions.csv',
    },
    {
        'dataset_id': 'personal_finance',
        'dataset_name': 'Personal Finance Dataset',
        'domain': 'personal_finance',
        'dataset_period': None,
        'source_path': raw_data_path / 'personal_finance' / 'Personal_Finance_Dataset.csv',
    },
]

assert len(DATASET_CATALOG) == 8, 'Catalog harus berisi tepat 8 dataset.'

catalog_preview = pd.DataFrame(
    {
        'dataset_id': item['dataset_id'],
        'dataset_name': item['dataset_name'],
        'domain': item['domain'],
        'dataset_period': item['dataset_period'],
        'source_path': str(item['source_path'].relative_to(project_root)),
        'file_exists': item['source_path'].exists(),
        'file_size_kb': round(item['source_path'].stat().st_size / 1024, 2) if item['source_path'].exists() else np.nan,
    }
    for item in DATASET_CATALOG
)

display(catalog_preview)

missing_files = catalog_preview.loc[~catalog_preview['file_exists'], 'source_path'].tolist()
if missing_files:
    raise FileNotFoundError(f'File raw tidak ditemukan: {missing_files}')

,dataset_id,dataset_name,domain,dataset_period,source_path,file_exists,file_size_kb
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,True,42.92
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,True,65.45
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,True,101.40
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,True,89.67
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,True,65.19
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,True,103.07
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,True,185.50
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,True,89.40


## Reusable Loader
Helper ini membaca CSV dan Excel. Jika `openpyxl` belum tersedia, file `.xlsx` tetap bisa dibaca memakai fallback XML dari struktur zip Excel.

In [3]:
SUPPORTED_EXTENSIONS = {'.csv', '.xlsx', '.xls'}
EXCEL_NAMESPACE = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}


def make_unique_columns(columns):
    counts = {}
    clean_columns = []

    for column in columns:
        text = '' if pd.isna(column) else str(column).strip()
        text = text or 'unnamed_column'
        base = text
        counts[base] = counts.get(base, 0) + 1
        if counts[base] > 1:
            text = f'{base}_{counts[base]}'
        clean_columns.append(text)

    return clean_columns


def excel_column_index(cell_reference):
    letters = ''.join(character for character in str(cell_reference) if character.isalpha()) or 'A'
    index = 0
    for letter in letters.upper():
        index = index * 26 + ord(letter) - ord('A') + 1
    return index - 1


def read_excel_without_engine(file_path):
    with ZipFile(file_path) as archive:
        names = set(archive.namelist())
        shared_strings = []

        if 'xl/sharedStrings.xml' in names:
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            shared_strings = [
                ''.join(text.text or '' for text in item.findall('.//main:t', EXCEL_NAMESPACE))
                for item in root.findall('main:si', EXCEL_NAMESPACE)
            ]

        worksheet_names = sorted(name for name in names if name.startswith('xl/worksheets/sheet') and name.endswith('.xml'))
        if not worksheet_names:
            return pd.DataFrame()

        sheet = ET.fromstring(archive.read(worksheet_names[0]))
        rows = []
        width = 0

        for row in sheet.findall('.//main:sheetData/main:row', EXCEL_NAMESPACE):
            values = []
            for cell in row.findall('main:c', EXCEL_NAMESPACE):
                column_index = excel_column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column_index:
                    values.append(np.nan)

                cell_type = cell.attrib.get('t')
                if cell_type == 'inlineStr':
                    value = ''.join(text.text or '' for text in cell.findall('.//main:t', EXCEL_NAMESPACE))
                else:
                    raw_value = cell.find('main:v', EXCEL_NAMESPACE)
                    value = raw_value.text if raw_value is not None else np.nan
                    if cell_type == 's' and pd.notna(value):
                        value = shared_strings[int(value)]

                values[column_index] = value

            width = max(width, len(values))
            rows.append(values)

    if not rows:
        return pd.DataFrame()

    rows = [row + [np.nan] * (width - len(row)) for row in rows]
    columns = make_unique_columns(rows[0])
    return pd.DataFrame(rows[1:], columns=columns).replace(r'^\s*$', np.nan, regex=True)


def read_data_file(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig').replace(r'^\s*$', np.nan, regex=True)

    if suffix in {'.xlsx', '.xls'}:
        try:
            return pd.read_excel(file_path).replace(r'^\s*$', np.nan, regex=True)
        except ImportError:
            return read_excel_without_engine(file_path)

    raise ValueError(f'Unsupported file extension: {file_path.suffix}')


def relative_to_project(path):
    path = Path(path)
    try:
        return str(path.relative_to(project_root))
    except ValueError:
        return str(path)

## Load Semua Dataset Secara Terpisah
Setiap dataset masuk ke dictionary `raw_datasets`. Tidak ada `pd.concat` untuk transaksi mentah.

In [4]:
raw_datasets = {}
manifest_rows = []

for meta in DATASET_CATALOG:
    dataset_id = meta['dataset_id']
    frame = read_data_file(meta['source_path']).copy()

    frame['_dataset_id'] = dataset_id
    frame['_dataset_name'] = meta['dataset_name']
    frame['_domain'] = meta['domain']
    frame['_dataset_period'] = meta['dataset_period']
    frame['_source_file'] = meta['source_path'].name
    frame['_source_path'] = relative_to_project(meta['source_path'])

    raw_datasets[dataset_id] = frame
    manifest_rows.append(
        {
            'dataset_id': dataset_id,
            'dataset_name': meta['dataset_name'],
            'domain': meta['domain'],
            'dataset_period': meta['dataset_period'],
            'source_path': relative_to_project(meta['source_path']),
            'record_count': int(len(frame)),
            'column_count': int(frame.shape[1]),
            'raw_output_path': relative_to_project(raw_separate_path / f'{dataset_id}.csv'),
        }
    )

    print(f"{dataset_id}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")

manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)

ecommerce_2024_01_january: 431 rows x 24 columns
ecommerce_2024_06_june: 697 rows x 24 columns
ecommerce_2024_12_december: 1,214 rows x 23 columns
ecommerce_2025_02_february: 957 rows x 24 columns
ecommerce_2025_07_july: 766 rows x 23 columns
ecommerce_2025_11_november: 1,131 rows x 24 columns
daily_household_transactions: 2,461 rows x 14 columns
personal_finance: 1,500 rows x 11 columns


,dataset_id,dataset_name,domain,dataset_period,source_path,record_count,column_count,raw_output_path
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,431,24,data/interim/raw_separate/ecommerce_2024_01_ja...
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,697,24,data/interim/raw_separate/ecommerce_2024_06_ju...
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,1214,23,data/interim/raw_separate/ecommerce_2024_12_de...
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,957,24,data/interim/raw_separate/ecommerce_2025_02_fe...
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,766,23,data/interim/raw_separate/ecommerce_2025_07_ju...
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,1131,24,data/interim/raw_separate/ecommerce_2025_11_no...
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,2461,14,data/interim/raw_separate/daily_household_tran...
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,1500,11,data/interim/raw_separate/personal_finance.csv


## Preview Struktur Tiap Dataset
Preview singkat ini membantu memastikan schema tiap dataset tetap terlihat sebelum masuk tahap assessing.

In [5]:
for dataset_id, frame in raw_datasets.items():
    print('\n' + '=' * 100)
    print(dataset_id)
    print(f'Shape: {frame.shape}')
    print('Columns:')
    print(list(frame.columns))
    display(frame.head(3))


ecommerce_2024_01_january
Shape: (431, 24)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...



ecommerce_2024_06_june
Shape: (697, 24)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0010173,4,4000,0,0,Other,1,Selesai,NaN,Kargo-J&T Cargo,Kartu Kredit/Debit,KOTA MEDAN,SUMATERA UTARA,25000,20000,242000,45000,2024-06-01 03:22,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...
1,ORD_0010174,3,1500,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-JNE Reguler,Online Payment,KOTA BEKASI,JAWA BARAT,10000,10000,72082,20000,2024-06-01 06:05,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...
2,ORD_0010175,1,500,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-JNE Reguler,SPayLater,KOTA JAKARTA TIMUR,DKI JAKARTA,0,10000,18069,10000,2024-06-01 06:26,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...



ecommerce_2024_12_december
Shape: (1214, 23)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


...

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0015042,1,500,0,0,Celengan,1,Selesai,NaN,SPX Standard,SPayLater,KOTA PALEMBANG,SUMATERA SELATAN,0,9900,18900,9900,2025-11-01 06:08,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...
1,ORD_0015043,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA MEDAN,SUMATERA UTARA,0,17000,31990,17000,2025-11-01 06:16,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...
2,ORD_0015044,1,6000,0,0,Nampan / Tray,1,Selesai,NaN,J&T Cargo,Saldo ShopeePay,KAB. WAY KANAN,LAMPUNG,0,54493,126900,54493,2025-11-01 06:41,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...



daily_household_transactions
Shape: (2461, 14)
Columns:
['Date', 'Mode', 'Category', 'Subcategory', 'Note', 'Amount', 'Income/Expense', 'Currency', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,20/09/2018 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...
1,20/09/2018 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...
2,19/09/2018,Saving Bank account 1,subscription,Netflix,1 month subscription,199.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...



personal_finance
Shape: (1500, 11)
Columns:
['Date', 'Transaction Description', 'Category', 'Amount', 'Type', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,Date,Transaction Description,Category,Amount,Type,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...


## Simpan Raw Dataset Terpisah
File output disimpan satu per satu di `data/interim/raw_separate/`. Manifest menjadi daftar acuan untuk notebook assessing dan cleaning.

In [6]:
for dataset_id, frame in raw_datasets.items():
    output_path = raw_separate_path / f'{dataset_id}.csv'
    frame.to_csv(output_path, index=False)
    print(f'Saved: {output_path.relative_to(project_root)} ({len(frame):,} rows)')

manifest_path = raw_separate_path / 'raw_dataset_manifest.json'
manifest_csv_path = raw_separate_path / 'raw_dataset_manifest.csv'

manifest_payload = manifest_df.to_dict(orient='records')
with open(manifest_path, 'w', encoding='utf-8') as file:
    json.dump(manifest_payload, file, indent=2)
manifest_df.to_csv(manifest_csv_path, index=False)

print('\nManifest saved:')
print(f'- {manifest_path.relative_to(project_root)}')
print(f'- {manifest_csv_path.relative_to(project_root)}')
print('\nCatatan: tidak ada dataset gabungan yang dibuat pada tahap gathering.')

Saved: data/interim/raw_separate/ecommerce_2024_01_january.csv (431 rows)
Saved: data/interim/raw_separate/ecommerce_2024_06_june.csv (697 rows)
Saved: data/interim/raw_separate/ecommerce_2024_12_december.csv (1,214 rows)
Saved: data/interim/raw_separate/ecommerce_2025_02_february.csv (957 rows)
Saved: data/interim/raw_separate/ecommerce_2025_07_july.csv (766 rows)
Saved: data/interim/raw_separate/ecommerce_2025_11_november.csv (1,131 rows)
Saved: data/interim/raw_separate/daily_household_transactions.csv (2,461 rows)
Saved: data/interim/raw_separate/personal_finance.csv (1,500 rows)

Manifest saved:
- data/interim/raw_separate/raw_dataset_manifest.json
- data/interim/raw_separate/raw_dataset_manifest.csv

Catatan: tidak ada dataset gabungan yang dibuat pada tahap gathering.


## Visualisasi Ringkas Hasil Gathering
Visualisasi ini memakai metadata per dataset, bukan gabungan transaksi mentah.

In [7]:
from html import escape


def build_bar_chart_html(frame, label_col, value_col, title, color, value_suffix='', value_format=',.0f'):
    chart = frame[[label_col, value_col]].copy()
    chart[value_col] = pd.to_numeric(chart[value_col], errors='coerce').fillna(0)
    max_value = chart[value_col].max() or 1
    rows = []

    for _, row in chart.iterrows():
        label = escape(str(row[label_col]))
        value = float(row[value_col])
        width = max(2, value / max_value * 100)
        value_text = format(value, value_format) + value_suffix
        rows.append(
            f'''<div class="bar-row">
                <div class="bar-label">{label}</div>
                <div class="bar-track"><div class="bar-fill" style="width:{width:.2f}%; background:{color};"></div></div>
                <div class="bar-value">{value_text}</div>
            </div>'''
        )

    return f'''<section class="chart-card">
        <h3>{escape(title)}</h3>
        {''.join(rows)}
    </section>'''


summary_df = manifest_df.sort_values('record_count', ascending=True).copy()
summary_df['short_name'] = summary_df['dataset_id'].str.replace('ecommerce_', 'ecom_', regex=False)
domain_df = manifest_df['domain'].value_counts().sort_values(ascending=True).rename_axis('domain').reset_index(name='dataset_count')

html = f'''
<style>
.dashboard-grid {{ display: grid; grid-template-columns: repeat(3, minmax(240px, 1fr)); gap: 16px; font-family: Arial, sans-serif; }}
.chart-card {{ border: 1px solid #d8dee4; border-radius: 8px; padding: 14px; background: #ffffff; }}
.chart-card h3 {{ margin: 0 0 12px 0; font-size: 16px; color: #1f2937; }}
.bar-row {{ display: grid; grid-template-columns: minmax(120px, 1.5fr) 2fr minmax(70px, .8fr); gap: 8px; align-items: center; margin: 8px 0; }}
.bar-label {{ font-size: 12px; color: #374151; overflow-wrap: anywhere; }}
.bar-track {{ height: 14px; background: #eef2f7; border-radius: 999px; overflow: hidden; }}
.bar-fill {{ height: 100%; border-radius: 999px; }}
.bar-value {{ font-size: 12px; color: #111827; text-align: right; font-variant-numeric: tabular-nums; }}
@media (max-width: 900px) {{ .dashboard-grid {{ grid-template-columns: 1fr; }} }}
</style>
<div class="dashboard-grid">
{build_bar_chart_html(summary_df, 'short_name', 'record_count', 'Jumlah Baris per Dataset', '#2f6f73')}
{build_bar_chart_html(summary_df, 'short_name', 'column_count', 'Jumlah Kolom per Dataset', '#bf7b45')}
{build_bar_chart_html(domain_df, 'domain', 'dataset_count', 'Jumlah Dataset per Domain', '#6f5f90')}
</div>
'''

dashboard_path = figures_path / '01_data_gathering_overview.html'
dashboard_path.write_text(html, encoding='utf-8')
display(HTML(html))
print(f'Dashboard saved: {dashboard_path.relative_to(project_root)}')


Dashboard saved: reports/figures/01_data_gathering_overview.html


## Output Gathering
Tahap gathering selesai ketika 8 dataset sudah terbaca dan tersimpan sebagai file terpisah. Notebook berikutnya (`02_data_assessing.ipynb`) membaca manifest ini untuk melakukan assessment per dataset.